# Phase 6: Machine Learning Pipeline

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score

from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
df = pd.read_csv("../data/Used_Car_Price_Prediction.csv")

In [3]:
# Columns to drop
columns_to_drop = [
    "registered_city",
    "registered_state",
    "rto",
    "source",
    "car_availability",
    "broker_quote",
    "original_price",
    "ad_created_on",
    "emi_starts_from",
    "booking_down_pymnt",
    "reserved",
    "is_hot",
    "times_viewed"
]

# Drop the columns
df = df.drop(columns=columns_to_drop)

# Check remaining columns
print("Remaining Columns:")
print(df.columns)

# Check dataset shape
print("\nDataset Shape:", df.shape)

Remaining Columns:
Index(['car_name', 'yr_mfr', 'fuel_type', 'kms_run', 'sale_price', 'city',
       'body_type', 'transmission', 'variant', 'assured_buy', 'make', 'model',
       'total_owners', 'car_rating', 'fitness_certificate', 'warranty_avail'],
      dtype='str')

Dataset Shape: (7400, 16)


In [4]:
# Drop high-cardinality and redundant columns
df = df.drop(columns=["car_name", "variant"])

# Check remaining columns
print(df.columns)

Index(['yr_mfr', 'fuel_type', 'kms_run', 'sale_price', 'city', 'body_type',
       'transmission', 'assured_buy', 'make', 'model', 'total_owners',
       'car_rating', 'fitness_certificate', 'warranty_avail'],
      dtype='str')


In [5]:
X = df.drop("sale_price", axis=1)
y = df["sale_price"]

In [6]:
categorical_features = X.select_dtypes(include=["object", "bool"]).columns

numerical_features = X.select_dtypes(include=["int64", "float64"]).columns

print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

Categorical Features:
Index(['fuel_type', 'city', 'body_type', 'transmission', 'assured_buy', 'make',
       'model', 'car_rating', 'fitness_certificate', 'warranty_avail'],
      dtype='str')

Numerical Features:
Index(['yr_mfr', 'kms_run', 'total_owners'], dtype='str')


C:\Users\hp\AppData\Local\Temp\ipykernel_22620\526590238.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object", "bool"]).columns


In [7]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [8]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42
    ))
])

In [9]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Train the Pipeline
pipeline.fit(X_train, y_train)

# Predictions
predictions = pipeline.predict(X_test)

# Evaluation
print("MAE:", mean_absolute_error(y_test, predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_test, predictions)))
print("R² Score:", r2_score(y_test, predictions))

MAE: 41291.52631531532
RMSE: 69846.04722746911
R² Score: 0.9299217716763531


In [10]:

import joblib

joblib.dump(pipeline, "../models/carvalue_pipeline.pkl")

print("✅ Pipeline saved successfully!")

✅ Pipeline saved successfully!


In [11]:
print(df.columns.tolist())

['yr_mfr', 'fuel_type', 'kms_run', 'sale_price', 'city', 'body_type', 'transmission', 'assured_buy', 'make', 'model', 'total_owners', 'car_rating', 'fitness_certificate', 'warranty_avail']
